# Code for extracting 1 lakh rows

In [ ]:
!pip -q install requests pandas

In [ ]:
import requests
import pandas as pd

In [ ]:
api_url = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

row_limit = 100000
batch_size = 10000

all_records = []

In [ ]:
for offset in range(0, row_limit, batch_size):
    parameters = {
        "$select": "unique_key,created_date,closed_date,agency,complaint_type,descriptor,borough,status,latitude,longitude",
        "$limit": batch_size,
        "$offset": offset,
        "$order": "created_date DESC"
    }

    response = requests.get(api_url, params=parameters, timeout=30)

    print("HTTP status:", response.status_code)
    response.raise_for_status()

    batch_records = response.json()
    all_records.extend(batch_records)

    print("Records collected so far:", len(all_records))

    if len(batch_records) < batch_size:
        print("No more records returned.")
        break

HTTP status: 200
Records collected so far: 10000
HTTP status: 200
Records collected so far: 20000
HTTP status: 200
Records collected so far: 30000
HTTP status: 200
Records collected so far: 40000
HTTP status: 200
Records collected so far: 50000
HTTP status: 200
Records collected so far: 60000
HTTP status: 200
Records collected so far: 70000
HTTP status: 200
Records collected so far: 80000
HTTP status: 200
Records collected so far: 90000
HTTP status: 200
Records collected so far: 100000


In [ ]:
nyc311_df = pd.DataFrame(all_records)

print("Rows collected:", len(nyc311_df))
print("Columns collected:", len(nyc311_df.columns))

nyc311_df.head()

Rows collected: 100000
Columns collected: 10


,unique_key,created_date,closed_date,agency,complaint_type,descriptor,borough,status,latitude,longitude
0,70919536,2026-10-29T02:38:41.000,2026-10-29T02:38:41.000,DOT,Street Condition,Pothole,MANHATTAN,Closed,40.794755363748415,-73.94234826097029
1,70919535,2026-10-29T02:35:24.000,NaN,DOT,Street Condition,Pothole,MANHATTAN,Open,40.794755363748415,-73.94234826097029
2,70925522,2026-10-29T02:03:51.000,NaN,NYPD,Noise - Street/Sidewalk,Loud Talking,MANHATTAN,In Progress,40.73637716939731,-73.98493484247116
3,70921261,2026-10-29T02:03:14.000,NaN,NYPD,Illegal Parking,Blocked Hydrant,BRONX,In Progress,40.853819743594265,-73.8312830766709
4,70919714,2026-10-29T02:01:47.000,NaN,DEP,Noise,Alarm,MANHATTAN,In Progress,40.74756297125971,-73.99684931958586


In [ ]:
print("Column names:")
print(nyc311_df.columns.tolist())

print("\nData types:")
print(nyc311_df.dtypes)

Column names:
['unique_key', 'created_date', 'closed_date', 'agency', 'complaint_type', 'descriptor', 'borough', 'status', 'latitude', 'longitude']

Data types:
unique_key        object
created_date      object
closed_date       object
agency            object
complaint_type    object
descriptor        object
borough           object
status            object
latitude          object
longitude         object
dtype: object


In [ ]:
csv_file = "nyc311_raw_sample.csv"

nyc311_df.to_csv(csv_file, index=False)

print("Saved file:", csv_file)

from google.colab import files
files.download(csv_file)

Saved file: nyc311_raw_sample.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Code for Extracting 1lakh rows - 23 fields

In [ ]:
import requests, pandas as pd, time
from datetime import datetime

api_url = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
row_target = 100000
batch_size = 10000

fields = ("unique_key,created_date,closed_date,due_date,agency,agency_name,"
          "complaint_type,descriptor,location_type,incident_zip,incident_address,"
          "street_name,city,borough,status,resolution_description,"
          "resolution_action_updated_date,community_board,open_data_channel_type,"
          "facility_type,park_borough,latitude,longitude")

# Fixed cutoff so new records arriving mid-download don't shift the pages
cutoff = datetime.now().strftime("%Y-%m-%dT00:00:00")
all_records = []

for offset in range(0, row_target, batch_size):
    params = {
        "$select": fields,
        "$where": f"created_date < '{cutoff}'",
        "$order": "created_date DESC, unique_key DESC",
        "$limit": batch_size,
        "$offset": offset,
    }
    for attempt in range(3):                      # retry transient failures
        try:
            r = requests.get(api_url, params=params, timeout=60)
            r.raise_for_status()
            batch = r.json()
            break
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt+1} failed at offset {offset}: {e}")
            time.sleep(5 * (attempt + 1))
    else:
        print("Giving up at offset", offset)
        break

    if not batch:
        break
    all_records.extend(batch)
    print(f"Offset {offset}: got {len(batch)} | total {len(all_records)}")
    if len(batch) < batch_size:
        break
    time.sleep(1)

nyc311_df = pd.DataFrame(all_records)
nyc311_df.to_csv("nyc311_raw.csv", index=False)

# Extraction log (a required deliverable item)
with open("extraction_log.txt", "w") as f:
    f.write(f"Extraction date: {datetime.now()}\nEndpoint: {api_url}\n"
            f"Filter: created_date < {cutoff}\nFields: {fields}\n"
            f"Target rows: {row_target}\nActual rows: {len(nyc311_df)}\n")

Offset 0: got 10000 | total 10000
Offset 10000: got 10000 | total 20000
Offset 20000: got 10000 | total 30000
Offset 30000: got 10000 | total 40000
Offset 40000: got 10000 | total 50000
Offset 50000: got 10000 | total 60000
Offset 60000: got 10000 | total 70000
Offset 70000: got 10000 | total 80000
Offset 80000: got 10000 | total 90000
Offset 90000: got 10000 | total 100000


In [ ]:
print("Rows collected:", len(nyc311_df))
print("Columns collected:", len(nyc311_df.columns))
display(nyc311_df.head())

Rows collected: 100000
Columns collected: 23


,unique_key,created_date,agency,agency_name,complaint_type,descriptor,incident_zip,city,borough,status,...,open_data_channel_type,facility_type,park_borough,latitude,longitude,incident_address,street_name,location_type,closed_date,due_date
0,70913336,2026-10-28T02:26:38.000,DOT,Department of Transportation,Street Condition,Pothole,11368,QUEENS,QUEENS,Open,...,UNKNOWN,N/A,QUEENS,40.74607475049938,-73.86407259580538,NaN,NaN,NaN,NaN,NaN
1,70917990,2026-10-28T02:16:03.000,DOT,Department of Transportation,Street Condition,Pothole,10473,BRONX,BRONX,Open,...,UNKNOWN,N/A,BRONX,NaN,NaN,HERMANY AVENUE,HERMANY AVENUE,NaN,NaN,NaN
2,70907416,2026-10-28T02:05:57.000,DEP,Department of Environmental Protection,Noise,Construction,10005,NEW YORK,MANHATTAN,In Progress,...,ONLINE,NaN,MANHATTAN,40.704755508056685,-74.008024928167,100 WATER STREET,WATER STREET,Building/Construction Site,NaN,NaN
3,70916726,2026-10-28T02:05:16.000,NYPD,New York City Police Department,Noise - Residential,Banging/Pounding,10461,BRONX,BRONX,In Progress,...,PHONE,NaN,BRONX,40.84937172684963,-73.83207510326014,2900 SAINT THERESA AVENUE,SAINT THERESA AVENUE,Residential Building/House,NaN,NaN
4,70911724,2026-10-28T02:05:13.000,DSNY,Department of Sanitation,Illegal Dumping,Removal Request,11236,BROOKLYN,BROOKLYN,In Progress,...,MOBILE,NaN,BROOKLYN,40.63961510590645,-73.90397500983536,1228 EAST 91 STREET,EAST 91 STREET,Street,NaN,NaN


In [ ]:
print("\nColumn names:")
print(nyc311_df.columns.tolist())
print("\nData types:")
print(nyc311_df.dtypes)


Column names:
['unique_key', 'created_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'incident_zip', 'city', 'borough', 'status', 'resolution_description', 'resolution_action_updated_date', 'community_board', 'open_data_channel_type', 'facility_type', 'park_borough', 'latitude', 'longitude', 'incident_address', 'street_name', 'location_type', 'closed_date', 'due_date']

Data types:
unique_key                        object
created_date                      object
agency                            object
agency_name                       object
complaint_type                    object
descriptor                        object
incident_zip                      object
city                              object
borough                           object
status                            object
resolution_description            object
resolution_action_updated_date    object
community_board                   object
open_data_channel_type            object
facility_type          

In [ ]:
print("\nMissing values per column:")
print(nyc311_df.isna().sum().sort_values(ascending=False))
print("\nDuplicate unique_key:", nyc311_df["unique_key"].duplicated().sum())
print("Date range:", nyc311_df["created_date"].min(), "to", nyc311_df["created_date"].max())


Missing values per column:
due_date                          99532
facility_type                     96887
closed_date                       29952
resolution_description             9900
resolution_action_updated_date     9776
location_type                      6328
city                               5719
street_name                        2410
incident_address                   2408
longitude                          1586
latitude                           1586
incident_zip                        873
created_date                          0
unique_key                            0
agency                                0
descriptor                            0
complaint_type                        0
agency_name                           0
status                                0
open_data_channel_type                0
community_board                       0
borough                               0
park_borough                          0
dtype: int64

Duplicate unique_key: 0
Date range: 20

In [ ]:
from google.colab import files
files.download("nyc311_raw.csv")
files.download("extraction_log.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


# Code for data extraction - monthly trend - Final


In [ ]:
import requests, pandas as pd, time
from datetime import datetime

api_url = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

fields = ("unique_key,created_date,closed_date,due_date,agency,agency_name,"
          "complaint_type,descriptor,location_type,incident_zip,incident_address,"
          "street_name,city,borough,status,resolution_description,"
          "resolution_action_updated_date,community_board,open_data_channel_type,"
          "facility_type,park_borough,latitude,longitude")

test_params = {
    "$select": fields,
    "$where": "created_date >= '2026-09-01T00:00:00' AND created_date < '2026-09-02T00:00:00' "
              "AND (unique_key like '%07' OR unique_key like '%37' OR unique_key like '%67')",
    "$limit": 5,
}
r = requests.get(api_url, params=test_params, timeout=60)
print("Status:", r.status_code)
print(r.text[:400] if r.status_code != 200 else f"OK, got {len(r.json())} rows")

Status: 200
OK, got 5 rows


In [ ]:
n_months = 12
suffixes = ["07", "37", "67"]        # 3 of every 100 ids = ~3% sample
id_filter = " OR ".join(f"unique_key like '%{s}'" for s in suffixes)

current_month = pd.Period(datetime.now(), "M")
months = pd.period_range(end=current_month - 1, periods=n_months, freq="M")

frames = []
for m in months:
    start = m.start_time.strftime("%Y-%m-%dT00:00:00")
    end = (m + 1).start_time.strftime("%Y-%m-%dT00:00:00")
    params = {
        "$select": fields,
        "$where": f"created_date >= '{start}' AND created_date < '{end}' AND ({id_filter})",
        "$order": "created_date DESC",
        "$limit": 50000,
    }
    batch = None
    for attempt in range(3):
        try:
            r = requests.get(api_url, params=params, timeout=120)
            if r.status_code == 400:
                print("400 error from API:", r.text[:400])
                raise SystemExit("Fix the query using the message above, then rerun")
            r.raise_for_status()
            batch = r.json()
            break
        except requests.exceptions.RequestException as e:
            print(f"{m}: attempt {attempt+1} failed: {e}")
            time.sleep(5 * (attempt + 1))
    if not batch:
        print(f"{m}: no data, skipped")
        continue
    if len(batch) == 50000:
        print(f"WARNING {m}: hit the 50,000 limit, month is truncated")
    frames.append(pd.DataFrame(batch))
    print(f"{m}: {len(batch)} rows")
    time.sleep(1)

nyc311_df = pd.concat(frames, ignore_index=True)
nyc311_df.to_csv("nyc311_raw.csv", index=False)

with open("extraction_log.txt", "w") as f:
    f.write(f"Extraction date: {datetime.now()}\n"
            f"Endpoint: {api_url}\n"
            f"Window: {months[0]} to {months[-1]} (last {n_months} full months)\n"
            f"Method: systematic ~3% sample, unique_key ending in {suffixes}, "
            f"same fraction applied to every month (no per-month cap)\n"
            f"Fields: {fields}\n"
            f"Actual rows: {len(nyc311_df)}\n")

print("\nTotal rows:", len(nyc311_df), "| Columns:", len(nyc311_df.columns))
print(nyc311_df["created_date"].min(), "to", nyc311_df["created_date"].max())

2025-10: 10062 rows
2025-11: 9152 rows
2025-12: 9962 rows
2026-01: 10444 rows
2026-02: 10031 rows
2026-03: 10281 rows
2026-04: 9085 rows
2026-05: 9961 rows
2026-06: 10051 rows
2026-07: 10282 rows
2026-08: 9843 rows
2026-09: 9712 rows

Total rows: 118866 | Columns: 23
2025-10-01T00:06:27.000 to 2026-09-30T23:57:00.000


In [ ]:
from google.colab import files
files.download("nyc311_raw.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Sample file

In [ ]:
nyc311_df.sample(20000, random_state=42).to_csv("nyc311_sample_20k.csv", index=False)
files.download("nyc311_sample_20k.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'extraction_log.txt', 'nyc311_raw Final.csv', 'nyc311_sample_20k.csv', 'nyc311_raw.csv', 'nyc311_raw_sample.csv', 'sample_data']


In [ ]:
os.rename("nyc311_raw Final.csv", "nyc311_raw.csv")